# Environment setup

**Destructive.** Drops and recreates `<catalog_prefix>_<env>` (bronze/silver/gold
schemas + checkpoints volumes) and the ingest schema (`landing` volume +
`source_manifest` table).

Run it by hand, never on a schedule:

```
databricks bundle run setup_job --target dev
```

In [ ]:
# Job parameters. Every notebook in this repo reads its config the same way, so
# nothing below is workspace specific and nothing hardcodes a catalog name.
configs = dict(dbutils.notebook.entry_point.getCurrentBindings())

ENV = configs.get("env", "dev")
CATALOG_PREFIX = configs.get("catalog_prefix", "rearc")

CATALOG = f"{CATALOG_PREFIX}_{ENV}"
INGEST_CATALOG = f"{CATALOG_PREFIX}_ingest"
INGEST_SCHEMA = ENV
LANDING_VOLUME = "landing"
LANDING_BASE = f"/Volumes/{INGEST_CATALOG}/{INGEST_SCHEMA}/{LANDING_VOLUME}"

SCHEMAS = ["bronze", "silver", "gold"]

print(f"ENV={ENV} | catalog={CATALOG} | ingest={INGEST_CATALOG}.{INGEST_SCHEMA} | landing={LANDING_BASE}")

## Reset

Everything below this point is safe to re-run; this cell is not.

In [0]:
spark.sql(f"DROP CATALOG IF EXISTS {CATALOG} CASCADE")
spark.sql(f"DROP SCHEMA IF EXISTS {INGEST_CATALOG}.{INGEST_SCHEMA} CASCADE")
print(f"dropped {CATALOG} and {INGEST_CATALOG}.{INGEST_SCHEMA}")

## Create catalogs, schemas, volumes

In [0]:
for catalog in [CATALOG, INGEST_CATALOG]:
    spark.sql(f"CREATE CATALOG IF NOT EXISTS {catalog}")

# One checkpoints volume per medallion schema: streaming checkpoints live in
# Unity Catalog volumes, never on DBFS.
for schema in SCHEMAS:
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {CATALOG}.{schema}")
    spark.sql(f"CREATE VOLUME IF NOT EXISTS {CATALOG}.{schema}.checkpoints")

# The ingest catalog stands in for upstream systems: one schema per environment,
# with a landing volume for file feeds.
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {INGEST_CATALOG}.{INGEST_SCHEMA}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {INGEST_CATALOG}.{INGEST_SCHEMA}.{LANDING_VOLUME}")

display(spark.sql(f"SHOW SCHEMAS IN {CATALOG}"))

## Ingest manifest table

Append-only ledger written by the `sourcing/` GitHub Actions fetcher (BLS +
DataUSA), never by anything running on Databricks compute. One row per
(source, dataset, ingest_ts) fetch attempt, even when unchanged, so the
fetcher can always find the most recent FETCHED row for conditional GET /
hash comparison.

Note: the Reset cell above drops this table along with the rest of the
ingest schema. That's fine, not destructive to anything downstream -- the
next fetcher run just treats every source as brand-new again (no
conditional header / no prior hash), which safely re-lands everything.

In [0]:
spark.sql(f"""
    CREATE TABLE IF NOT EXISTS {INGEST_CATALOG}.{INGEST_SCHEMA}.source_manifest (
        source          STRING,
        dataset         STRING,
        ingest_ts       STRING,
        status          STRING,
        http_status     INT,
        content_sha256  STRING,
        last_modified   STRING,
        bytes           BIGINT,
        source_url      STRING,
        landing_path    STRING,
        run_id          STRING,
        fetched_at      TIMESTAMP,
        error_message   STRING
    )
    TBLPROPERTIES ('delta.appendOnly' = 'true')
""")
print(f"  {INGEST_CATALOG}.{INGEST_SCHEMA}.source_manifest ready (append-only)")

In [ ]:
print("=" * 60)
print(f"{CATALOG} ready")
print("=" * 60)
for schema in SCHEMAS:
    print(f"  {CATALOG}.{schema}")
print(f"  {INGEST_CATALOG}.{INGEST_SCHEMA}.source_manifest")
print(f"  {LANDING_BASE}")
print()
print("Next: uv run python -m sourcing --env <env>, then")
print("      databricks bundle run declarative_pipeline_silver_gold --target <env>")